# 00 — Ghost Artist Research (Kaggle Only)
**No API calls required.** Mines ghost candidates from 114K tracks using audio feature variance.

**Logic:** Ghost factory artists use the same AI model with fixed settings → every track sounds identical → very low catalog variance across danceability, energy, valence, acousticness.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

FIGURES = Path('../paper/figures')
FIGURES.mkdir(parents=True, exist_ok=True)
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

FEATURE_4D = ['danceability', 'energy', 'valence', 'acousticness']

# Known Swedish ghost production companies from DN investigation
KNOWN_GHOST_COMPANIES = [
    'Firefly Entertainment AB',
    'Lucille AB',
    'Tombola Music',
    'Catfish Music Group',
    'Calm and Collected Music Publishing',
]

print('Setup OK')

## 1. Load Kaggle Dataset

In [ ]:
df = pd.read_csv('../data/kaggle/dataset.csv')
print(f'Loaded: {len(df):,} tracks, {df["artists"].nunique():,} unique artists')
df.head(2)

## 2. Compute Per-Artist Statistics

In [ ]:
# Only artists with 10+ tracks — need enough data for variance to be meaningful
artist_counts = df['artists'].value_counts()
artists_10plus = artist_counts[artist_counts >= 10].index
df_filtered = df[df['artists'].isin(artists_10plus)]
print(f'Artists with 10+ tracks: {len(artists_10plus):,}')
print(f'Tracks in that subset:   {len(df_filtered):,}')

rows = []
for artist, grp in df_filtered.groupby('artists'):
    row = {
        'artist': artist,
        'track_count': len(grp),
        'mean_duration_ms': float(grp['duration_ms'].mean()),
        'genres': ','.join(sorted(grp['track_genre'].dropna().unique().tolist())),
        'genre_count': grp['track_genre'].nunique(),
    }
    for f in FEATURE_4D:
        row[f'var_{f}'] = float(grp[f].var())
        row[f'mean_{f}'] = float(grp[f].mean())
    row['total_variance'] = sum(row[f'var_{f}'] for f in FEATURE_4D)
    rows.append(row)

artist_stats = pd.DataFrame(rows)
print(f'\nArtist stats computed for {len(artist_stats):,} artists')
artist_stats.head(3)

## 3. Flag Ghost Candidates

In [ ]:
# Ghost rules:
#   total_variance < 0.01   — very uniform catalog
#   track_count > 20        — large catalog (maximizing streams)
#   mean_duration_ms 60K–180K — 1–3 min (streaming-optimized short tracks)

ghost_mask = (
    (artist_stats['total_variance'] < 0.01) &
    (artist_stats['track_count'] > 20) &
    (artist_stats['mean_duration_ms'] >= 60_000) &
    (artist_stats['mean_duration_ms'] <= 180_000)
)

ghost_candidates = artist_stats[ghost_mask].sort_values('total_variance').copy()
ghost_candidates['flag'] = 'GHOST_CANDIDATE'

# Organic controls: high variance, reasonable track count
organic_mask = (
    (artist_stats['total_variance'] > 0.08) &
    (artist_stats['track_count'] >= 10) &
    (artist_stats['track_count'] <= 200)
)
organic_controls = artist_stats[organic_mask].sort_values('total_variance', ascending=False).copy()
organic_controls['flag'] = 'ORGANIC_CONTROL'

print(f'Ghost candidates:  {len(ghost_candidates):,}')
print(f'Organic controls:  {len(organic_controls):,}')

In [ ]:
print('\nTop 50 ghost candidates (sorted by total_variance ascending):')
display_cols = ['artist','track_count','total_variance','mean_duration_ms','genre_count','genres']
print(ghost_candidates.head(50)[display_cols].to_string(index=False))

In [ ]:
# Visualization: scatter of total_variance vs track_count
fig, ax = plt.subplots(figsize=(12, 7))

# All artists
ax.scatter(artist_stats['total_variance'], artist_stats['track_count'],
           alpha=0.15, s=8, color='gray', label='All artists')

# Ghost candidates
ax.scatter(ghost_candidates['total_variance'], ghost_candidates['track_count'],
           alpha=0.7, s=30, color='#e74c3c', label=f'Ghost candidates ({len(ghost_candidates)})')

# Organic controls
ax.scatter(organic_controls['total_variance'], organic_controls['track_count'],
           alpha=0.5, s=20, color='#2ecc71', label=f'Organic controls ({len(organic_controls)})')

# Label top 10 ghost candidates
for _, row in ghost_candidates.head(10).iterrows():
    ax.annotate(row['artist'], (row['total_variance'], row['track_count']),
                fontsize=7, alpha=0.8,
                xytext=(5, 5), textcoords='offset points')

ax.axvline(0.01, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Ghost threshold (0.01)')
ax.set_xlabel('Total Variance (4D audio feature space)', fontsize=12)
ax.set_ylabel('Track Count', fontsize=12)
ax.set_title('Ghost Candidate Detection via Audio Feature Variance\n'
             'Red = ghost candidates (low variance + large catalog)', fontsize=13, fontweight='bold')
ax.set_xlim(-0.005, 0.5)
ax.set_ylim(0, artist_stats['track_count'].quantile(0.99))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES / 'ghost_candidates_scatter.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: ghost_candidates_scatter.png')

In [ ]:
# Per-feature variance comparison: ghost vs organic
if len(ghost_candidates) > 0 and len(organic_controls) > 0:
    fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=False)

    for ax, feat in zip(axes, FEATURE_4D):
        ghost_vals = ghost_candidates[f'var_{feat}'].values
        organic_vals = organic_controls[f'var_{feat}'].sample(min(len(ghost_candidates)*2, len(organic_controls)),
                                                               random_state=42).values
        ax.hist(ghost_vals, bins=20, alpha=0.7, color='#e74c3c', label='Ghost candidates', density=True)
        ax.hist(organic_vals, bins=20, alpha=0.7, color='#2ecc71', label='Organic controls', density=True)
        ax.set_title(feat.capitalize(), fontweight='bold')
        ax.set_xlabel('Variance')
        ax.set_ylabel('Density')
        if feat == FEATURE_4D[0]:
            ax.legend(fontsize=8)

    plt.suptitle('Per-Feature Variance: Ghost Candidates vs Organic Controls\n'
                 'Ghost artists cluster near zero in all dimensions', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES / 'ghost_vs_organic_variance.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved: ghost_vs_organic_variance.png')

## 4. Cross-Reference with Known Ghost Companies

In [ ]:
# Check if any ghost candidates appear in Kaggle genres that match known ghost labels
ghost_genres = ['sleep', 'ambient', 'meditation', 'study', 'focus', 'relax', 'chill', 'white-noise']

matching = ghost_candidates[
    ghost_candidates['genres'].str.lower().str.contains('|'.join(ghost_genres), na=False)
]

print(f'Ghost candidates in suspicious genres ({", ".join(ghost_genres)}):')
print(f'Count: {len(matching)}')
if not matching.empty:
    print(matching[['artist','track_count','total_variance','genres']].head(20).to_string(index=False))
else:
    print('None found — Kaggle genre labels may not match exactly')
    print()
    print('Ghost candidate genres breakdown:')
    from collections import Counter
    all_genres = []
    for g in ghost_candidates['genres'].dropna():
        all_genres.extend(g.split(','))
    genre_counts = Counter(all_genres)
    for genre, count in genre_counts.most_common(15):
        print(f'  {genre:<30} {count}')

## 5. Save Outputs

In [ ]:
ghost_candidates.to_csv(PROCESSED / 'ghost_candidates_kaggle.csv', index=False)
organic_controls.to_csv(PROCESSED / 'organic_controls_kaggle.csv', index=False)

print(f'Saved ghost_candidates_kaggle.csv  ({len(ghost_candidates)} candidates)')
print(f'Saved organic_controls_kaggle.csv  ({len(organic_controls)} controls)')

print()
print('=== SUMMARY ===')
print(f'Found {len(ghost_candidates)} ghost candidates and {len(organic_controls)} organic controls from Kaggle dataset')
print()
print('Top 10 ghost candidates:')
for _, r in ghost_candidates.head(10).iterrows():
    print(f'  {r.artist:<40} var={r.total_variance:.5f}  tracks={int(r.track_count)}  avg_dur={int(r.mean_duration_ms/1000)}s')

print()
print('Top 5 organic controls:')
for _, r in organic_controls.head(5).iterrows():
    print(f'  {r.artist:<40} var={r.total_variance:.4f}  tracks={int(r.track_count)}')